# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [7]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO = "transacciones_sample.csv"
SOLUCION = "q5_solucion.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 5000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/08 21:03,14,811143E30,146477,8112133A0,6160.23,Mexican Peso,6160.23,Mexican Peso,Cheque,0
1,2022/09/01 14:11,26922,80330D020,27356,81A02AFC0,16946.26,Yuan,16946.26,Yuan,Credit Card,0
2,2022/09/10 08:15,9735,8042FDA10,19931,804EB7BC0,1772.13,Yuan,1772.13,Yuan,Cash,0
3,2022/09/07 14:16,19836,80642DBB0,217073,8064D5D00,73361.71,Yen,73361.71,Yen,Cheque,0
4,2022/09/02 07:43,24633,801EFE020,17256,81C11F940,179.87,US Dollar,179.87,US Dollar,Cheque,0


In [8]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05" (inclusive)
INICIO = "2022/09/01"
FIN    = "2022/09/06"  # "2022/09/06" para incluir todo el día 5/9

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] < FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()

Transacciones en el período: 2759


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
1,2022/09/01 14:11,26922,80330D020,27356,81A02AFC0,16946.26,Yuan,16946.26,Yuan,Credit Card,0
4,2022/09/02 07:43,24633,801EFE020,17256,81C11F940,179.87,US Dollar,179.87,US Dollar,Cheque,0
5,2022/09/01 12:19,2692,806973D60,2692,806973D60,16855.24,US Dollar,16855.24,US Dollar,Reinvestment,0
8,2022/09/01 11:42,29158,806A974C0,29158,806A974C0,10780.01,US Dollar,10780.01,US Dollar,Reinvestment,0
9,2022/09/02 20:05,70,10042B6F0,19521,8049B6840,1761.57,Yuan,1761.57,Yuan,Cheque,0


In [9]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 382


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
18,2022/09/04 09:47,12,805C54960,115648,805C5D970,312068.83,Yen,312068.83,Yen,ACH,0
23,2022/09/01 07:58,328899,8118A8600,15516,8118AA970,15118.63,Euro,15118.63,Euro,ACH,0
46,2022/09/02 01:12,38944,8167B17D0,11756,8167B2750,659.58,Yuan,659.58,Yuan,ACH,0
49,2022/09/02 16:19,1110,800C77D10,13243,80FF45820,293.30,US Dollar,293.30,US Dollar,Wire,0
68,2022/09/05 21:18,25634,8071FD240,25788,812D197F0,128.61,Euro,128.61,Euro,Wire,0


In [10]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [11]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 3


,Timestamp,Receiving Currency,Amount Received,amount_usd
1081,2022/09/01 09:47,Rupee,77.75,0.976910
2805,2022/09/05 08:39,Rupee,31.71,0.397009
4144,2022/09/01 23:06,Rupee,53.73,0.675105


In [12]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv(SOLUCION, index=False)
print(f"Guardado en {SOLUCION}")


Q5 resultado: 3 transacciones
Guardado en q5_solucion.csv
